# Baseline models

Use the common CLI outputs to compare a prior-probability Dummy and regularized Logistic Regression. All displayed metrics use the same independent model-selection partition; threshold-dependent metrics use 0.5. Average Precision is the step-wise AP metric, not trapezoidal PR area.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
METRICS = ROOT / "reports" / "metrics"
FIGURES = ROOT / "reports" / "figures"
assert (METRICS / "final_test_metrics.json").exists(), "Run the full training CLI first."


def read_json(name):
    return json.loads((METRICS / name).read_text())


def show_figure(name):
    display(Image(filename=str(FIGURES / name)))

In [2]:
comparison = pd.read_csv(METRICS / "model_comparison.csv")
baselines = comparison[comparison.family.isin(["dummy", "logistic"])]
display(
    baselines[
        [
            "model",
            "roc_auc",
            "average_precision",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "brier_score",
            "log_loss",
            "threshold",
        ]
    ]
)

,model,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1,brier_score,log_loss,threshold
4,logistic_engineered,0.854829,0.378707,0.803147,0.775485,0.217273,0.743539,0.336281,0.146019,0.470271,0.5
5,logistic_raw,0.852444,0.376165,0.803547,0.775700,0.217653,0.743539,0.336735,0.145901,0.483143,0.5
6,dummy,0.500000,0.067070,0.932930,0.500000,0.000000,0.000000,0.000000,0.062571,0.245993,0.5


## Why accuracy is insufficient

The Dummy predicts the training event prevalence as a probability and makes no positive classifications at threshold 0.5. Its accuracy reflects class imbalance, while recall and Average Precision expose its limitations.

In [3]:
dummy = baselines[baselines.model == "dummy"].iloc[0]
print(
    f"Dummy accuracy {dummy.accuracy:.3%}, recall {dummy.recall:.3%}, "
    f"AP {dummy.average_precision:.5f}."
)
display(
    pd.read_csv(METRICS / "search_logistic.csv")[
        ["params", "mean_test_ap", "std_test_ap", "mean_test_auc", "mean_test_brier"]
    ]
)

Dummy accuracy 93.293%, recall 0.000%, AP 0.06707.


,params,mean_test_ap,std_test_ap,mean_test_auc,mean_test_brier
0,"{'model__class_weight': None, 'model__C': 0.05}",0.377056,0.010566,0.849268,-0.050809
1,"{'model__class_weight': 'balanced', 'model__C'...",0.382397,0.011522,0.853990,-0.145198
2,"{'model__class_weight': 'balanced', 'model__C'...",0.382405,0.011522,0.853985,-0.145185


## Coefficients

Coefficients below refer to the **engineered Logistic Regression** candidate, after median imputation, missing indicators, and standardization. A coefficient is a change in model log-odds per standardized unit, conditional on correlated features. It is not a causal effect or a calibrated probability change.

In [4]:
coefficients = pd.read_csv(METRICS / "logistic_coefficients.csv")
display(
    coefficients.assign(magnitude=coefficients.standardized_coefficient.abs())
    .sort_values("magnitude", ascending=False)
    .head(15)
)

,feature,standardized_coefficient,magnitude
17,log_utilization,0.682917,0.682917
12,has_past_due,0.379235,0.379235
1,age,-0.282791,0.282791
7,NumberRealEstateLoansOrLines,0.241263,0.241263
4,MonthlyIncome,-0.233502,0.233502
13,has_severe_delinquency,0.198452,0.198452
5,NumberOfOpenCreditLinesAndLoans,0.155197,0.155197
8,NumberOfTime60-89DaysPastDueNotWorse,0.134419,0.134419
19,real_estate_share,-0.106486,0.106486
10,total_past_due,0.098082,0.098082


Logistic Regression improves ranking over the Dummy. Class weighting can increase recall at 0.5 while distorting probability calibration; separate threshold optimization and calibration are therefore evaluated later.